In [1]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

# SAM 3 Agent

This notebook shows an example of how an MLLM can use SAM 3 as a tool, i.e., "SAM 3 Agent", to segment more complex text queries such as "the leftmost child wearing blue vest".

## Env Setup

First install `sam3` in your environment using the [installation instructions](https://github.com/facebookresearch/sam3?tab=readme-ov-file#installation) in the repository.

In [ ]:
import torch
# turn on tfloat32 for Ampere GPUs
# https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook. If your card doesn't support it, try float16 instead
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

# inference mode for the whole notebook. Disable if you need gradients
torch.inference_mode().__enter__()

In [ ]:
import os

SAM3_ROOT = os.path.dirname(os.getcwd())
os.chdir(SAM3_ROOT)

# setup GPU to use -  A single GPU is good with the purpose of this demo
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
_ = os.system("nvidia-smi")

## Build SAM3 Model

In [ ]:
import sam3
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")
bpe_path = f"{sam3_root}/assets/bpe_simple_vocab_16e6.txt.gz"
model = build_sam3_image_model(bpe_path=bpe_path)
processor = Sam3Processor(model, confidence_threshold=0.5)

## LLM Setup

Config which MLLM to use, it can either be a model served by vLLM that you launch from your own machine or a model is served via external API. If you want to using a vLLM model, we also provided insturctions below.

In [ ]:
LLM_CONFIGS = {
    # vLLM-served models
    "qwen3_vl_8b_thinking": {
        "provider": "vllm",
        "model": "Qwen/Qwen3-VL-8B-Thinking",
    },
    # ---------------------------------------------------------------------------
    # Gemma 4 via Ollama — runs fully locally on a laptop, no API key required.
    #
    # Prerequisites:
    #   1. Install Ollama:         https://ollama.com
    #   2. Pull a Gemma 4 model:   ollama pull gemma4:12b
    #   3. Start Ollama server:    ollama serve
    #
    # Variants (choose based on available RAM):
    #   gemma4:4b   ~4 GB RAM   fastest
    #   gemma4:12b  ~8 GB RAM   recommended
    #   gemma4:27b  ~20 GB RAM  best multimodal quality
    # ---------------------------------------------------------------------------
    "gemma4_12b_local": {
        "provider": "ollama",
        "model": "gemma4:12b",
        "ollama_host": "http://localhost:11434",
    },
    "gemma4_27b_local": {
        "provider": "ollama",
        "model": "gemma4:27b",
        "ollama_host": "http://localhost:11434",
    },
    # models served via external APIs
    # add your own
}

model = "qwen3_vl_8b_thinking"  # change to "gemma4_12b_local" to run locally
LLM_API_KEY = "DUMMY_API_KEY"

llm_config = LLM_CONFIGS[model]
llm_config["api_key"] = LLM_API_KEY
llm_config["name"] = model

# setup API endpoint
if llm_config["provider"] == "vllm":
    LLM_SERVER_URL = "http://0.0.0.0:8001/v1"  # replace with your vLLM server address
elif llm_config["provider"] == "ollama":
    LLM_SERVER_URL = llm_config["ollama_host"] + "/v1"
else:
    LLM_SERVER_URL = llm_config["base_url"]


### Setup LLM server

#### Option A — Ollama (local, laptop-friendly, no API key needed)

Run Gemma 4 directly on your machine:
```bash
# Install Ollama: https://ollama.com
ollama pull gemma4:12b   # ~8 GB RAM – recommended
# ollama pull gemma4:27b  # ~20 GB RAM – best multimodal quality
ollama serve             # usually starts automatically
```
Then set `model = "gemma4_12b_local"` in the cell above.

#### Option B — vLLM (GPU server)

Only required for vLLM-served models; skip if using Ollama or a cloud API.

```bash
conda create -n vllm python=3.12
pip install vllm --extra-index-url https://download.pytorch.org/whl/cu128
# Start the server:
vllm serve Qwen/Qwen3-VL-8B-Thinking --tensor-parallel-size 4 \
    --allowed-local-media-path / --enforce-eager --port 8001
```


## Run SAM3 Agent Inference

In [ ]:
from functools import partial
from IPython.display import display, Image
from sam3.agent.client_llm import (
    send_generate_request as send_generate_request_orig,
    send_generate_request_ollama,
)
from sam3.agent.client_sam3 import call_sam_service as call_sam_service_orig
from sam3.agent.inference import run_single_image_inference


In [ ]:
# prepare input args and run single image inference
image = "assets/images/test_image.jpg"
prompt = "the leftmost child wearing blue vest"
image = os.path.abspath(image)

if llm_config["provider"] == "ollama":
    # Local Gemma 4 via Ollama — no API key needed
    send_generate_request = partial(
        send_generate_request_ollama,
        model=llm_config["model"],
        ollama_host=llm_config["ollama_host"],
    )
else:
    # vLLM / cloud API
    send_generate_request = partial(
        send_generate_request_orig,
        server_url=LLM_SERVER_URL,
        model=llm_config["model"],
        api_key=llm_config["api_key"],
    )

call_sam_service = partial(call_sam_service_orig, sam3_processor=processor)
output_image_path = run_single_image_inference(
    image, prompt, llm_config, send_generate_request, call_sam_service,
    debug=True, output_dir="agent_output"
)

# display output
if output_image_path is not None:
    display(Image(filename=output_image_path))
